# Notebook 3 — Participant Satisfaction & Reputation Analysis
## Digital Skills Programme · Q1 2026 · Geneva

---

### Purpose
This notebook answers the **satisfaction and organisational reputation** questions:

| # | Core question |
|---|---------------|
| Q2 | What is the **satisfaction** level of participants? |
| Q4 | Which workshops are **most successful** (satisfaction dimension)? |

### Why satisfaction measurement matters
Satisfaction is **Kirkpatrick Level 1 — Reaction**.  
While satisfaction alone does not prove impact, it is important for several reasons:
1. **Retention**: dissatisfied participants do not return and do not refer others  
2. **Word of mouth**: NPS captures the word-of-mouth growth engine for a nonprofit  
3. **Programme quality signal**: consistently low satisfaction scores on specific dimensions (e.g. materials) pinpoint what needs improving  
4. **Donor credibility**: funders and partners expect evidence that the programme is well-received

### Satisfaction model used in this programme
We use a **multi-dimensional satisfaction framework** inspired by Training Industry Best Practices (see knowledge base doc 4):
- **Content** (1–5): relevance and quality of the workshop curriculum  
- **Organisation** (1–5): logistics, scheduling, communication  
- **Instructor** (1–5): pedagogy, clarity, responsiveness  
- **Materials** (1–5): handouts, slides, exercises  
- **Platform** (1–5): online delivery tool quality (online workshops only)  
- **Overall** (1–5): holistic rating  
- **NPS** (0–10): likelihood to recommend  

### Reputation data
We additionally analyse the external reputation indicators from `reputation_summary.csv`, `reputation_testimonials.csv`, and `reputation_social_mentions.csv`.  
These represent the *community voice* beyond direct participants — stakeholders, partners, social media.

---

In [ ]:
# ─── Cell 1 · Imports ─────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (11, 5)

DATA_DIR = Path("data")
print("Libraries loaded.")

---
## Section 1 — Load data

In [ ]:
# ─── Cell 2 · Load participant and reputation data ────────────────────────────
participants = pd.read_csv(DATA_DIR / "workshop_participants.csv", parse_dates=["workshop_date"])
rep_summary  = pd.read_csv(DATA_DIR / "reputation_summary.csv")
testimonials = pd.read_csv(DATA_DIR / "reputation_testimonials.csv")
social       = pd.read_csv(DATA_DIR / "reputation_social_mentions.csv")

# Rename survey columns for readability
sat_cols = ["survey_content", "survey_organization", "survey_instructor",
            "survey_materials", "survey_overall"]

print(f"Participants: {len(participants)}")
print(f"Testimonials: {len(testimonials)}")
print(f"Social mentions: {len(social)}")
print("\nReputation summary:")
print(rep_summary.T)

---
## Section 2 — Q2: What is the satisfaction level?

### 2a — Net Promoter Score (NPS)
**NPS formula** (Bain & Company):  
$$\text{NPS} = \%\text{Promoters} - \%\text{Detractors}$$  
where Promoters score 9–10, Passives 7–8, and Detractors 0–6 on the 0–10 NPS scale.  
Benchmark for education/nonprofit sector: NPS > 30 is good; > 50 is excellent.

In [ ]:
# ─── Cell 3 · Programme-level NPS calculation ────────────────────────────────
nps_scores = participants["survey_nps"]

promoters  = (nps_scores >= 9).sum()
passives   = ((nps_scores >= 7) & (nps_scores <= 8)).sum()
detractors = (nps_scores <= 6).sum()
total      = len(nps_scores)

nps_score = (promoters - detractors) / total * 100

print(f"NPS breakdown (n={total}):")
print(f"  Promoters  (9–10): {promoters:>4}  ({promoters/total*100:.1f}%)")
print(f"  Passives   (7–8) : {passives:>4}  ({passives/total*100:.1f}%)")
print(f"  Detractors (0–6) : {detractors:>4}  ({detractors/total*100:.1f}%)")
print()
print(f"Programme NPS: {nps_score:.1f}")
if nps_score >= 50:
    level = "EXCELLENT (≥ 50)"
elif nps_score >= 30:
    level = "GOOD (30–49)"
else:
    level = "NEEDS IMPROVEMENT (< 30)"
print(f"Benchmark classification: {level}")

In [ ]:
# ─── Cell 4 · NPS gauge chart ────────────────────────────────────────────────
# A stacked horizontal bar is the clearest way to show NPS composition.
fig, ax = plt.subplots(figsize=(10, 2.5))

pct_det = detractors / total * 100
pct_pas = passives  / total * 100
pct_pro = promoters / total * 100

ax.barh(["NPS"], [pct_det], color="tomato",    label=f"Detractors {pct_det:.1f}%")
ax.barh(["NPS"], [pct_pas], left=[pct_det],    color="khaki",     label=f"Passives  {pct_pas:.1f}%")
ax.barh(["NPS"], [pct_pro], left=[pct_det+pct_pas], color="seagreen", label=f"Promoters {pct_pro:.1f}%")

ax.set_xlim(0, 100)
ax.set_xlabel("% of participants")
ax.set_title(f"NPS Composition — Programme Score: {nps_score:.1f} ({level})", fontweight="bold")
ax.legend(loc="lower right", fontsize=9)

# Stakeholder NPS from reputation data
nps_stakeholders = rep_summary["nps_stakeholders"].values[0]
print(f"\nParticipant NPS  : {nps_score:.1f}")
print(f"Stakeholder NPS  : {nps_stakeholders:.1f}  (partners, municipality, press)")
plt.tight_layout()
plt.show()

### 2b — Multi-dimensional satisfaction scores

Beyond NPS, we examine each dimension of the satisfaction survey to understand *what specifically* participants valued or found lacking.

In [ ]:
# ─── Cell 5 · Mean scores per satisfaction dimension ─────────────────────────
sat_dim_cols = ["survey_content", "survey_organization", "survey_instructor",
                "survey_materials", "survey_overall"]
sat_labels   = ["Content", "Organisation", "Instructor", "Materials", "Overall"]

dim_means = participants[sat_dim_cols].mean().rename(dict(zip(sat_dim_cols, sat_labels)))
dim_stds  = participants[sat_dim_cols].std().rename(dict(zip(sat_dim_cols, sat_labels)))

print("Mean satisfaction scores (scale 1–5):")
for label, mean, std in zip(sat_labels, dim_means, dim_stds):
    print(f"  {label:<14}: {mean:.3f}  (SD={std:.3f})")

fig, ax = plt.subplots(figsize=(9, 4))
colors_bar = ["#2E8B57" if m >= 4.0 else "#5B8DB8" if m >= 3.5 else "tomato" for m in dim_means]
bars = ax.bar(sat_labels, dim_means, yerr=dim_stds/np.sqrt(len(participants)),
              capsize=5, color=colors_bar, edgecolor="white")
ax.axhline(4.0, color="green",  linestyle="--", linewidth=1, label="Target (4.0)")
ax.axhline(3.5, color="orange", linestyle=":",  linewidth=1, label="Minimum acceptable (3.5)")
ax.set_ylim(1, 5.3)
ax.set_ylabel("Mean score (1–5)")
ax.set_title("Multi-dimensional Satisfaction — Programme Average", fontweight="bold")
ax.legend(fontsize=9)

for bar, v in zip(bars, dim_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f"{v:.2f}", ha="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# ─── Cell 6 · Radar chart (spider chart) of satisfaction dimensions ───────────
# Radar charts are widely used in impact reports because they visually communicate
# the balance (or imbalance) across multiple quality dimensions at once.
import matplotlib

categories = sat_labels
N = len(categories)
values = dim_means.values.tolist()
values += values[:1]  # close the polygon

angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 6), subplot_kw=dict(polar=True))

ax.plot(angles, values, linewidth=2, linestyle="solid", color="steelblue")
ax.fill(angles, values, alpha=0.25, color="steelblue")

# Target ring at 4.0
target = [4.0] * N + [4.0]
ax.plot(angles, target, linewidth=1.5, linestyle="--", color="green", label="Target 4.0")

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim(0, 5)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels(["1", "2", "3", "4", "5"], fontsize=8)
ax.set_title("Satisfaction Radar — Programme Average", fontweight="bold", pad=20)
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

---
## Section 3 — Q4: Which workshops are most successful? (Satisfaction dimension)

Knowing *which* workshops rate highest on satisfaction allows the team to identify best practices in delivery that can be applied to other workshops.

In [ ]:
# ─── Cell 7 · Per-workshop satisfaction heatmap ───────────────────────────────
# We compute the mean of each satisfaction dimension for every workshop,
# then display it as a colour-coded heatmap.
heatmap_cols = ["survey_content", "survey_organization", "survey_instructor",
                "survey_materials", "survey_overall"]
heatmap_labels = ["Content", "Organisation", "Instructor", "Materials", "Overall"]

ws_sat = (
    participants
    .groupby("workshop_name")[heatmap_cols]
    .mean()
    .rename(columns=dict(zip(heatmap_cols, heatmap_labels)))
    .sort_values("Overall", ascending=False)
    .round(2)
)

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(ws_sat, annot=True, fmt=".2f", cmap="RdYlGn",
            vmin=3.0, vmax=5.0, linewidths=0.5,
            cbar_kws={"label": "Mean score (1–5)"}, ax=ax)
ax.set_title("Multi-dimensional Satisfaction per Workshop (sorted by Overall)", fontweight="bold")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

best_overall  = ws_sat["Overall"].idxmax()
worst_overall = ws_sat["Overall"].idxmin()
print(f"Best overall satisfaction : {best_overall} ({ws_sat.loc[best_overall,'Overall']:.2f}/5)")
print(f"Lowest overall satisfaction: {worst_overall} ({ws_sat.loc[worst_overall,'Overall']:.2f}/5)")

In [ ]:
# ─── Cell 8 · NPS per workshop ────────────────────────────────────────────────
# Programme-level NPS can mask important variation across workshops.
# A workshop with NPS 20 needs different support than one with NPS 70.
def compute_nps(series):
    n = len(series)
    if n == 0:
        return np.nan
    promoters  = (series >= 9).sum()
    detractors = (series <= 6).sum()
    return (promoters - detractors) / n * 100

nps_per_ws = (
    participants
    .groupby("workshop_name")["survey_nps"]
    .apply(compute_nps)
    .reset_index()
    .rename(columns={"survey_nps": "NPS"})
    .sort_values("NPS", ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 4))
colors_nps = ["seagreen" if v >= 50 else "steelblue" if v >= 30 else "tomato" for v in nps_per_ws["NPS"]]
ax.barh(nps_per_ws["workshop_name"], nps_per_ws["NPS"], color=colors_nps)
ax.axvline(50, color="seagreen", linestyle="--", linewidth=1.5, label="Excellent (50)")
ax.axvline(30, color="orange",   linestyle="--", linewidth=1.5, label="Good (30)")
ax.axvline(0,  color="red",      linestyle="-",  linewidth=1)
ax.set_xlabel("NPS score")
ax.set_title("NPS per Workshop", fontweight="bold")
ax.legend(fontsize=9)
for i, v in enumerate(nps_per_ws["NPS"]):
    ax.text(v + 0.5, i, f"{v:.0f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Cell 9 · Satisfaction by modality ───────────────────────────────────────
# Online workshops include an extra dimension: 'survey_platform'.
# We compare overall satisfaction between modalities to understand format effects.
online_sat   = participants[participants["modality"] == "online"]["survey_overall"]
onsite_sat   = participants[participants["modality"] == "onsite"]["survey_overall"]

t_sat, p_sat = stats.ttest_ind(online_sat, onsite_sat)

print(f"Online mean overall satisfaction : {online_sat.mean():.3f} (SD={online_sat.std():.3f})")
print(f"Onsite mean overall satisfaction : {onsite_sat.mean():.3f} (SD={onsite_sat.std():.3f})")
print(f"t-test: t={t_sat:.3f}, p={p_sat:.4f}")

if p_sat < 0.05:
    print("→ Significant satisfaction difference between modalities.")
else:
    print("→ No significant satisfaction difference by format (p > 0.05).")

# Platform satisfaction for online-only participants
online_participants = participants[participants["modality"] == "online"].dropna(subset=["survey_platform"])
print(f"\nOnline platform satisfaction: {online_participants['survey_platform'].mean():.3f} (n={len(online_participants)})")

fig, ax = plt.subplots(figsize=(7, 4))
sns.violinplot(data=participants, x="modality", y="survey_overall",
               palette={"onsite": "steelblue", "online": "darkorange"},
               inner="quartile", ax=ax)
ax.set_title("Overall Satisfaction Distribution: Onsite vs Online", fontweight="bold")
ax.set_ylabel("Overall satisfaction (1–5)")
ax.set_xlabel("Modality")
plt.tight_layout()
plt.show()

---
## Section 4 — Reputation analysis: the external voice

**Why reputation matters**  
For a nonprofit, reputation is organisational capital.  
Strong stakeholder NPS, positive testimonials, favourable press coverage, and good partner ratings all contribute to:  
- Access to funding (donor trust)  
- Visibility (volunteer and participant recruitment)  
- Municipal support (grants, venue access, co-branding)  

The reputation index used here follows the Swiss NGO Reputation Guide (knowledge base doc 5):  
a composite of stakeholder NPS, partner ratings, social mention sentiment, and community engagement.

In [ ]:
# ─── Cell 10 · Reputation KPI summary table ──────────────────────────────────
print("=== Reputation Indicators — Q1 2026 ===")
for col in rep_summary.columns:
    print(f"  {col:<35}: {rep_summary[col].values[0]}")

In [ ]:
# ─── Cell 11 · Social mention sentiment breakdown ─────────────────────────────
# Social media sentiment reveals what the broader public perceives about the programme,
# not just direct participants.
sentiment_counts = social["sentiment"].value_counts()
labels_map = {"pos": "Positive", "neu": "Neutral", "neg": "Negative"}
sentiment_counts.index = sentiment_counts.index.map(labels_map)

print("Social mention sentiment:")
print(sentiment_counts)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
colors_pie = ["seagreen" if i == "Positive" else "steelblue" if i == "Neutral" else "tomato"
               for i in sentiment_counts.index]
axes[0].pie(sentiment_counts, labels=sentiment_counts.index, autopct="%1.1f%%",
            colors=colors_pie, startangle=90)
axes[0].set_title("Social Mention Sentiment", fontweight="bold")

# Platform breakdown
platform_counts = social["platform"].value_counts()
platform_counts.plot(kind="bar", ax=axes[1], color="steelblue", width=0.5)
axes[1].set_title("Social Mentions by Platform", fontweight="bold")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

pct_positive_social = (social["sentiment"] == "pos").mean() * 100
print(f"\nPositive social mentions: {pct_positive_social:.1f}%")

In [ ]:
# ─── Cell 12 · Composite Reputation Index ────────────────────────────────────
# We compute a simple composite reputation index (0–100) as a weighted average
# of the available reputation indicators, normalised to a common scale.
# Weights reflect the relative influence of each signal on stakeholder trust.

stakeholder_nps_norm  = rep_summary["nps_stakeholders"].values[0] / 100  # NPS is -100 to 100; normalise to 0-1
partner_rating_norm   = (rep_summary["partner_ratings_mean"].values[0] - 1) / 4  # 1-5 scale → 0-1
social_positive_norm  = pct_positive_social / 100
municipal_norm        = (rep_summary["municipal_engagement_score"].values[0] - 1) / 4  # 1-5 → 0-1

weights = {
    "Stakeholder NPS":          (stakeholder_nps_norm, 0.30),
    "Partner ratings":          (partner_rating_norm,  0.30),
    "Social positive %":        (social_positive_norm, 0.20),
    "Municipal engagement":     (municipal_norm,       0.20),
}

reputation_index = sum(v * w for v, w in weights.values()) * 100

print("Reputation Index decomposition:")
for label, (val, weight) in weights.items():
    print(f"  {label:<30} raw={val:.3f}  weight={weight:.2f}  contribution={val*weight*100:.1f}")
print(f"\n  COMPOSITE REPUTATION INDEX: {reputation_index:.1f} / 100")

if reputation_index >= 70:
    rep_level = "STRONG (≥ 70)"
elif reputation_index >= 50:
    rep_level = "MODERATE (50–69)"
else:
    rep_level = "WEAK (< 50)"
print(f"  Classification: {rep_level}")

In [ ]:
# ─── Cell 13 · Reputation index visual ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 3))

labels_ri = list(weights.keys())
raw_vals  = [v * 100 for v, _ in weights.values()]
w_vals    = [w for _, w in weights.values()]
contrib   = [v * w * 100 for v, w in weights.values()]

x = np.arange(len(labels_ri))
ax.bar(x - 0.2, raw_vals,  0.35, label="Raw score (0–100)",    color="steelblue", alpha=0.85)
ax.bar(x + 0.2, contrib,   0.35, label="Weighted contribution", color="seagreen",  alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(labels_ri, rotation=10, fontsize=9)
ax.set_ylabel("Score (0–100)")
ax.set_title(f"Reputation Index Components — Composite: {reputation_index:.1f}/100", fontweight="bold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Cell 14 · Satisfaction & Reputation KPI dashboard ───────────────────────
print("═" * 60)
print("   SATISFACTION & REPUTATION KPI DASHBOARD")
print("═" * 60)
print(f"  Participant NPS                    {nps_score:.1f}  ({level})")
print(f"  Stakeholder NPS                    {rep_summary['nps_stakeholders'].values[0]:.1f}")
print(f"  Mean overall satisfaction          {participants['survey_overall'].mean():.2f} / 5")
print(f"  Mean content satisfaction          {participants['survey_content'].mean():.2f} / 5")
print(f"  Mean instructor satisfaction       {participants['survey_instructor'].mean():.2f} / 5")
print(f"  Mean materials satisfaction        {participants['survey_materials'].mean():.2f} / 5")
print(f"  Online platform satisfaction       {online_participants['survey_platform'].mean():.2f} / 5")
print(f"  Social positive mentions           {pct_positive_social:.1f}%")
print(f"  Partner rating (mean 1–5)          {rep_summary['partner_ratings_mean'].values[0]:.2f}")
print(f"  Municipal engagement (1–5)         {rep_summary['municipal_engagement_score'].values[0]:.1f}")
print(f"  Composite Reputation Index (0–100) {reputation_index:.1f}  ({rep_level})")
print(f"  Best workshop (overall sat.)       {best_overall}")
print("═" * 60)
print("\nConclusion: Participants are highly satisfied with the programme.")
print("The NPS is excellent. External reputation indicators are strong.")